# 📗 질문 유형에 맞는 검색 경로를 선택합니다

**Jev로 질문 유형을 판단하고 필요한 검색만 실행합니다.**

<img src="images/12_question_search_routing.png" width="1100" alt="Jev가 질문을 분류한 뒤 하이브리드·그래프·복합 경로 중 하나로 검색하고 근거를 인용하는 흐름">


- 줄거리 중심의 일반 질문: 하이브리드 검색 → 원문 근거 → 답변
- 그래프 조회 질문: Text-to-Cypher → 조회 결과 → 답변
- 줄거리와 그래프 조건을 함께 묻는 복합 질문: 질문 분리 → 전문·벡터·그래프 검색 → 조건 적용 → 근거 → 답변

Jev가 `hybrid`·`graph`·`combined` 중 하나를 선택합니다. 질문 분리는 복합 질문에서만 실행합니다.
복합 경로는 RRF 결과 중 그래프 조건에 맞는 영화만 남기고, 그래프에서만 찾은 영화는 뒤에 보완합니다.
최종 답변은 선택한 원문의 줄거리까지 확인해 작성합니다.
전문 검색을 사용하는 경로는 기존 Jev 언어 판단도 실행합니다.
분리 결과에서 원 질문의 조건이 빠지거나 새 조건이 생기지 않았는지 확인하세요.

> Sleepless in Seattle과 출연 배우를 최소 2명 공유하고 1993년 이후(1993년 제외) 개봉한 영화 중,
> 서로 경쟁하는 서점 주인들이 익명 이메일로 사랑하게 되는 작품을 찾아 주세요.

`You've Got Mail`의 원문과 `Tom Hanks`·`Meg Ryan`의 출연 관계가 판단 근거입니다.
`Joe Versus the Volcano`는 공유 배우만 보고 포함하면 안 됩니다. 개봉연도 조건도 확인하세요.
이 정보는 순위 결과가 아니라 **기존 데이터에서 확인할 기준**입니다.

**작성 방법:** 구분선 안의 `변수명 = ...`에서 우변 전체를 직접 작성하세요. 구분선 밖 코드는 그대로 실행합니다. 긴 프롬프트·출력 형식·결과 표시는 제공됩니다.


## 1. 실행 환경과 기존 그래프를 준비합니다

아래 한 셀에서 라이브러리·모델·DB를 연결하고 영화 그래프·원문·배포 벡터를 적재합니다.
문서 임베딩은 다시 호출하지 않습니다. 같은 검색 대상의 기존 인덱스는 재생성하며 노드와 벡터는 유지합니다.
`Chunk.embedding` 공용 인덱스도 정리하므로 이전 실습에서 다시 쓰려면 해당 인덱스를 재생성하세요.


In [ ]:
# 검색·DB·모델에 필요한 라이브러리를 불러옵니다.
import json
import os
import re
import sys
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from neo4j import GraphDatabase
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_typesafe import Choice, TypeSafeClassifier
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from neo4j_graphrag.retrievers import VectorRetriever
from neo4j_graphrag.types import RetrieverResultItem
from pydantic import BaseModel, Field

from langchain_neo4j import GraphCypherQAChain, Neo4jGraph
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda
from langchain_classic.retrievers import EnsembleRetriever


# 노트북 기준 경로입니다. 정답은 상위 폴더를 사용합니다.
material_dir = Path(".")
data_dir = material_dir / "data"
sys.path.insert(0, str(material_dir.resolve()))
from graph_data import load_graph, read_vectors


# 공통 설정을 읽고 교안의 .env를 우선 적용합니다.
load_dotenv(find_dotenv(usecwd=True))
load_dotenv(material_dir.resolve().parent / ".env")
load_dotenv(material_dir / ".env", override=True)
# 모델만 준비합니다. API 호출은 검색·답변 단계에서 합니다.
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-large", dimensions=768,
    check_embedding_ctx_length=False,  # 준비한 청크를 자동으로 나누지 않습니다.
)
llm = ChatOpenAI(model=os.getenv("OPENAI_CHAT_MODEL", "gpt-5.6-luna"))

# .env의 주소·계정으로 Neo4j에 연결합니다.
neo4j_uri = os.environ["NEO4J_URI"]
driver = GraphDatabase.driver(
    neo4j_uri, auth=(os.environ["NEO4J_USER"], os.environ["NEO4J_PASSWORD"])
)
driver.verify_connectivity()
address = urlparse(neo4j_uri)
print("Neo4j 연결:", address.hostname, address.port or 7687)


def run_cypher(query, **params):
    """Cypher 결과를 행 딕셔너리의 리스트로 반환합니다."""
    # 인자를 Cypher의 $파라미터로 전달합니다.
    with driver.session() as session:
        # 조회 결과를 딕셔너리 목록으로 반환합니다.
        return [record.data() for record in session.run(query, **params)]


# 영화 그래프·원문과 기존 청크 벡터를 불러옵니다.
movies = load_graph(data_dir / "movies_extraction_packet.json")
chunk_ids, vectors = read_vectors(
    data_dir / "embeddings/movies_complete.npz", movies["chunks"], embedding_model
)
# 검색 결과의 원문 ID로 줄거리를 찾을 사전입니다.
documents_by_id = movies["documents"]
print("개체:", len(movies["nodes"]), "관계:", len(movies["relations"]))
print("문서:", len(documents_by_id), "청크:", len(chunk_ids), "벡터:", vectors.shape)


# 이전 실습의 공용 Chunk.embedding 벡터 인덱스만 정리합니다.
# 노드·관계·저장된 임베딩은 유지됩니다.
indexes = run_cypher("""
    SHOW VECTOR INDEXES
    YIELD name, labelsOrTypes, properties
    WHERE labelsOrTypes = ['Chunk'] AND properties = ['embedding']
    RETURN name
""")

for index in indexes:
    name = index["name"].replace("`", "``")
    run_cypher(f"DROP INDEX `{name}` IF EXISTS")
    print("삭제:", index["name"])

# 개체의 standard_id 중복을 막는 고유 제약을 만듭니다.
run_cypher("CREATE CONSTRAINT rag_entity_id IF NOT EXISTS FOR (n:RAGEntity) REQUIRE n.standard_id IS UNIQUE")
# 원문 id 중복을 막는 고유 제약을 만듭니다.
run_cypher("CREATE CONSTRAINT document_id IF NOT EXISTS FOR (n:Document) REQUIRE n.id IS UNIQUE")
# 청크 id 중복을 막는 고유 제약을 만듭니다.
run_cypher("CREATE CONSTRAINT chunk_id IF NOT EXISTS FOR (n:Chunk) REQUIRE n.id IS UNIQUE")

# 영화·인물을 종류별로 적재합니다.
for entity_type in ["Movie", "Person"]:
    # 현재 종류의 개체만 고릅니다.
    rows = [row for row in movies["nodes"] if row["entity_type"] == entity_type]
    # 현재 종류의 영화·인물 노드를 만들고 속성을 갱신합니다.
    run_cypher(f"""
        // 목록을 개체별 행으로 펼칩니다.
        UNWIND $rows AS item
        // 같은 ID의 노드를 재사용하고, 없으면 생성합니다.
        MERGE (n:RAGEntity {{standard_id: item.standard_id}})
        // 종류 레이블을 추가하고 속성·이름을 갱신합니다.
        SET n:{entity_type}, n += item.properties, n.name = item.name
    """, rows=rows)

# 적재한 노드 사이에 종류별 관계를 연결합니다.
for relation in ["ACTED_IN", "DIRECTED", "PRODUCED", "WROTE", "REVIEWED", "FOLLOWS"]:
    # 현재 유형의 관계만 고릅니다.
    rows = [row for row in movies["relations"] if row["relation"] == relation]
    # 현재 유형의 관계를 연결하고 근거·출처를 저장합니다.
    run_cypher(f"""
        // 목록을 관계별 행으로 펼칩니다.
        UNWIND $rows AS item
        // 관계의 시작·끝 노드를 ID로 찾습니다.
        MATCH (a:RAGEntity {{standard_id: item.subject_id}})
        MATCH (b:RAGEntity {{standard_id: item.object_id}})
        // 같은 유형·방향·근거 ID의 관계가 없으면 만들고, 있으면 재사용합니다.
        MERGE (a)-[r:{relation} {{claim_id: item.claim_id}}]->(b)
        // 관계 속성과 근거·출처를 저장합니다.
        SET r += item.properties, r.evidence = item.evidence,
            r.source_url = item.source_url, r.source_doc_id = item.source_doc_id
    """, rows=rows)


# 원문 노드를 먼저 저장합니다.
run_cypher("""
    // 원문 목록을 한 문서당 한 행으로 펼칩니다.
    UNWIND $rows AS item
    // 같은 ID의 원문 노드가 없으면 만들고, 있으면 재사용합니다.
    MERGE (d:Document {id: item.doc_id})
    // 제목·본문·출처 URL을 갱신합니다.
    SET d.title = item.title, d.text = item.text, d.url = item.url
""", rows=list(documents_by_id.values()))

# 원문과 해당 영화를 연결합니다.
run_cypher("""
    // doc_id로 원문을, standard_id로 해당 영화를 찾습니다.
    UNWIND $rows AS item
    MATCH (d:Document {id: item.doc_id})
    MATCH (m:Movie {standard_id: item.standard_id})
    // 원문과 해당 영화를 ABOUT_MOVIE 관계로 연결합니다.
    MERGE (d)-[:ABOUT_MOVIE]->(m)
""", rows=movies["document_links"])

# 청크와 벡터의 순서를 맞춰 전체 저장 목록을 만듭니다.
rows = []
for index, chunk in enumerate(movies["chunks"]):
    # NumPy 벡터를 DB에 전달할 목록으로 바꿉니다.
    rows.append({"id": chunk_ids[index], "doc_id": chunk.metadata["source_doc_id"],
                 "text": chunk.page_content, "embedding": vectors[index].tolist()})

# 전체 청크와 벡터를 한 번에 저장합니다.
run_cypher("""
    UNWIND $rows AS item
    // 원문을 찾고 청크를 ID로 생성·재사용합니다.
    MATCH (d:Document {id: item.doc_id})
    MERGE (c:Chunk {id: item.id})
    // 이 교안의 영화 청크만 검색하도록 전용 레이블을 붙입니다.
    SET c:Day49MovieChunk
    // text는 전문 검색, embedding은 벡터 검색, source_id는 원문별 결과 통합에 씁니다.
    SET c.text = item.text, c.source_id = item.doc_id, c.embedding = item.embedding
    // 청크와 원문을 FROM_DOCUMENT 관계로 연결합니다.
    MERGE (c)-[:FROM_DOCUMENT]->(d)
""", rows=rows)


# 같은 이름의 옛 설정과 같은 검색 대상의 다른 이름 인덱스를 정리합니다.
# IF NOT EXISTS만 쓰면 다른 이름의 인덱스 때문에 필요한 이름이 생성되지 않을 수 있습니다.
indexes = run_cypher("""
    SHOW INDEXES YIELD name, type, entityType, labelsOrTypes, properties
    WHERE name IN ['movie_chunks', 'movie_fulltext']
       OR (entityType = 'NODE' AND labelsOrTypes = ['Day49MovieChunk']
           AND ((type = 'VECTOR' AND properties = ['embedding'])
             OR (type = 'FULLTEXT' AND properties = ['text'])))
    RETURN name
""")
for index in indexes:
    name = index["name"].replace("`", "``")
    run_cypher(f"DROP INDEX `{name}` IF EXISTS")
# 아래에서 정해진 이름·차원·분석기로 다시 만듭니다. 노드와 저장된 벡터는 유지됩니다.

# 벡터 유사도 검색용 인덱스를 만듭니다.
run_cypher("""
    CREATE VECTOR INDEX movie_chunks IF NOT EXISTS
    // 이 교안의 영화 청크만 벡터 검색 대상으로 지정합니다.
    FOR (c:Day49MovieChunk) ON c.embedding
    // 저장된 벡터의 768차원과 코사인 유사도를 설정합니다.
    OPTIONS {indexConfig: {`vector.dimensions`: 768, `vector.similarity_function`: 'cosine'}}
""")
# 벡터 인덱스가 검색 가능한 상태가 될 때까지 최대 300초 기다립니다.
run_cypher("CALL db.awaitIndex('movie_chunks', 300)")
# 단어 일치 검색용 전문 인덱스를 만듭니다.
run_cypher("""
    CREATE FULLTEXT INDEX movie_fulltext IF NOT EXISTS
    // 같은 영화 청크의 text만 전문 검색 대상으로 지정합니다.
    FOR (c:Day49MovieChunk) ON EACH [c.text]
    // 영어 분석기를 사용하고, 본문 변경을 커밋 시 인덱스에 반영합니다.
    OPTIONS {indexConfig: {`fulltext.analyzer`: 'english', `fulltext.eventually_consistent`: false}}
""")
# 전문 인덱스가 검색 가능한 상태가 될 때까지 최대 300초 기다립니다.
run_cypher("CALL db.awaitIndex('movie_fulltext', 300)")
print("기존 문서 임베딩을 적재했습니다. 문서 임베딩 API 호출은 없습니다.")


## 2. 공통 검색·근거 코드를 준비합니다

벡터·전문 검색, RRF 결합과 근거 처리 함수를 한 셀로 준비합니다.
질문 분류·분리·Text-to-Cypher·답변 체인은 다음 실습에서 만듭니다.
이 셀은 함수와 체인을 정의하며 질문에 대한 검색·모델 호출은 실습에서 실행합니다.


In [ ]:
def format_vector(record):
    """검색된 청크의 본문과 원문 ID·유사도 점수를 반환합니다."""
    node = record["node"]
    # content는 청크 본문, metadata는 청크 ID·원문 ID·유사도를 담는 사전입니다.
    return RetrieverResultItem(content=node["text"], metadata={
        "chunk_id": node["id"], "source_id": node["source_id"], "score": record["score"]
    })


# 벡터 인덱스와 결과 변환 함수를 연결해 벡터 검색기를 만듭니다.
vector_retriever = VectorRetriever(
    driver, index_name="movie_chunks", embedder=embedding_model,
    result_formatter=format_vector,
)


def vector_documents(query_vector, top_k=12):
    """검색된 청크 중 영화별 최고 점수 한 건을 대표 후보로 반환합니다."""
    # 질문 벡터와 유사한 청크를 점수 내림차순으로 검색합니다.
    result = vector_retriever.search(query_vector=query_vector, top_k=top_k)
    # 영화별 후보 목록을 만들기 위해 처음 나온 최고 점수 청크를 대표로 남깁니다.
    rows, seen = [], set()
    for item in result.items:
        row = item.metadata
        if row["source_id"] not in seen:
            seen.add(row["source_id"])
            rows.append(dict(row))
    return rows


def show_documents(rows):
    """전달받은 모든 검색 결과를 순서대로 조회해 제목과 본문 앞부분을 보여 줍니다."""
    display(pd.DataFrame(rows))
    # 검색 순위를 함께 전달해 DB 조회 후에도 같은 순서를 유지합니다.
    ranked_ids = [{"rank": rank, "source_id": row["source_id"]}
                  for rank, row in enumerate(rows)]
    # 전달받은 원문 전체의 제목과 본문을 Neo4j에서 한 번에 조회합니다.
    documents = run_cypher("""
        UNWIND $rows AS item
        // 검색 결과의 원문 ID로 저장된 문서를 찾습니다.
        MATCH (d:Document {id: item.source_id})
        RETURN d.id AS source_id, d.title AS title, d.text AS text
        // DB의 반환 순서를 원래 검색 순위에 맞춥니다.
        ORDER BY item.rank
    """, rows=ranked_ids)
    for doc in documents:
        print("\n제목:", doc["title"], "| 원문 ID:", doc["source_id"])
        # 긴 원문은 앞 650자만 출력합니다.
        print(doc["text"][:650])


# Jev가 한글·영어 중 질문의 언어를 선택하도록 설정합니다.
language_classifier = TypeSafeClassifier(model=os.getenv("TYPESAFE_MODEL", "jev-1.13.0"))
language_question = Choice(
    instructions="질문의 주된 언어를 선택하세요.",
    criteria={"ko": "한글 질문. 영어 영화명 등이 섞인 한글 문장도 포함합니다.",
              "en": "한글 없이 영어 단어나 문장으로 작성된 질문입니다."},
)


# 영어 검색어를 문자열 목록으로 받습니다.
class EnglishKeywords(BaseModel):
    """영어 원문 검색에 사용할 핵심어 목록입니다."""
    keywords: list[str] = Field(description="중복 없는 영어 단어 4~8개")


# 한글 질문에서 영어 검색어 목록을 만드는 체인입니다.
lexical_prompt = ChatPromptTemplate.from_messages([
    ("system", "한글 영화 줄거리 질문을 영어 원문 검색용 핵심어 4~8개로 바꾸세요. "
               "질문의 핵심 사건·대상을 나타내는 단어를 골라 keywords의 각 항목에 영어 단어 하나씩 담으세요. "
               "영화 정답이나 배우를 추측해 추가하지 마세요. 질문에 없는 고유명사는 넣지 마세요."),
    ("human", "{question}"),
])
lexical_chain = lexical_prompt | llm.with_structured_output(EnglishKeywords)


def fulltext_documents(query, top_k=12):
    """질문의 언어에 맞춰 전문 검색하고 원문별 최고 점수 청크를 반환합니다."""
    # state에 질문, questions에 언어 선택 기준을 전달합니다.
    decision = language_classifier.invoke({
        "state": query, "questions": {"language": language_question},
    })
    language = decision.choices["language"].choice
    # 한글은 영어 검색어 목록을 받아 OR로 연결하고, 영어는 그대로 사용합니다.
    if language == "ko":
        terms = lexical_chain.invoke({"question": query})
        search_text = " OR ".join(f'"{word}"' for word in terms.keywords)
    else:
        search_text = query
    print("질문 언어:", language, "| 전문 검색어:", search_text)
    # 전문 인덱스에서 검색어와 일치하는 청크를 최대 top_k개 조회합니다.
    hits = run_cypher("""
        // 전문 인덱스에서 관련 청크를 최대 top_k개 조회합니다.
        CALL db.index.fulltext.queryNodes('movie_fulltext', $search_text, {limit: $top_k})
        YIELD node, score
        // 결과 통합에 쓸 원문 ID를 반환합니다.
        RETURN node.source_id AS source_id, node.id AS chunk_id, score
        // 점수 내림차순으로 정렬합니다.
        ORDER BY score DESC, chunk_id
    """, search_text=search_text, top_k=top_k)
    rows, seen = [], set()
    # 영화별 순위를 비교하도록 원문마다 최고 점수 검색 행을 대표로 남깁니다.
    for row in hits:
        source_id = row["source_id"]
        if source_id not in seen:
            seen.add(source_id)
            rows.append(row)
    return rows




def as_documents(rows):
    """원문별 검색 결과를 LangChain Document 목록으로 바꿉니다."""
    return [Document(page_content=documents_by_id[row["source_id"]]["text"],
                     metadata={"source_id": row["source_id"]}) for row in rows]


# 기존 검색 함수를 LangChain에서 호출할 수 있도록 연결합니다.
# 두 함수는 이미 원문별 중복을 제거한 결과를 반환합니다.
fulltext_retriever = RunnableLambda(
    lambda query: as_documents(fulltext_documents(query, top_k=30))
)
dense_retriever = RunnableLambda(
    lambda query: as_documents(vector_documents(embedding_model.embed_query(query), top_k=30))
)
# 전문·벡터 검색만 가중 RRF로 결합합니다. 같은 원문은 source_id로 합칩니다.
ensemble_retriever = EnsembleRetriever(
    retrievers=[fulltext_retriever, dense_retriever],
    weights=[0.4, 0.6], c=60, id_key="source_id",
)

# 실제 저장된 관계 ID로 근거 문장을 찾습니다.
relations_by_id = {row["claim_id"]: row for row in movies["relations"]}


def collect_evidence(graph_rows, source_ids=()):
    """선택한 원문·그래프 조회 사실·관계 근거를 하나의 목록으로 모읍니다."""
    evidence = []
    for source_id in source_ids:
        doc = documents_by_id[source_id]
        evidence.append({"evidence_id": source_id, "kind": "plot", "title": doc["title"],
                         "url": doc["url"], "text": doc["text"]})
    claim_ids = set()
    for i, row in enumerate(graph_rows, start=1):
        evidence.append({"evidence_id": f"graph:{i}", "kind": "graph_record",
                         "text": json.dumps(row["facts"], ensure_ascii=False)})
        claim_ids.update(row["claim_ids"])
    for claim_id in sorted(claim_ids):
        evidence.append({"evidence_id": claim_id, "kind": "relation",
                         "text": relations_by_id[claim_id]["evidence"]})
    return evidence


## 3. 질문 유형에 따라 검색하고 답변합니다

아래 질문을 `new_question`에 넣어 경로를 비교합니다. 먼저 예시 복합 질문으로 실행한 뒤 다른 질문으로 바꾸세요.
질문을 바꾸면 **경로 판단 셀부터 마지막 답변 셀까지** 다시 실행합니다.

| 유형 | 예시 질문 | 예상 경로 · 실행할 검색 |
|---|---|---|
| 일반 | 서로 경쟁하는 서점 주인들이 익명 이메일로 사랑하게 되는 영화는 무엇인가요? | `hybrid` · 전문 + 벡터 |
| 그래프 조회 | The Matrix의 개봉연도와 출연 배우 이름을 알려 주세요. | `graph` · Text-to-Cypher |
| 복합(관계·연도·줄거리) | Sleepless in Seattle과 출연 배우를 최소 2명 공유하고 1993년 이후(1993년 제외) 개봉한 다른 영화 중 서로 경쟁하는 서점 주인들이 익명 이메일로 사랑하게 되는 작품을 찾아 공통 배우와 근거를 보여 주세요. | `combined` · 전문 + 벡터 + 그래프 |
| 복합(연도·줄거리) | 2000년 이후(2000년 포함)에 개봉한 영화 중 비행기 추락 뒤 무인도에서 혼자 생존하는 이야기를 찾아 주세요. | `combined` · 전문 + 벡터 + 그래프 |

복합 질문에 배우 조건이 반드시 필요한 것은 아닙니다. 줄거리와 그래프 속성 조건만 있어도 `combined`입니다.


### 🖐️ 함께 따라하기: Jev가 검색 경로를 선택하도록 설정합니다

**제공 코드:** `Choice`로 만든 `route_question`의 분류 지시문과 세 경로 기준을 제공합니다.

**작성할 부분:**

- `route_classifier`를 `TypeSafeClassifier`로 만드세요. `model`은 환경변수 `TYPESAFE_MODEL`을 사용하고 기본값은 `jev-1.13.0`입니다.


In [ ]:
# 질문 유형 판단도 기존 언어 판단과 같은 Jev 모델을 사용합니다.
# ====================================================================
route_classifier = ...
# ====================================================================
route_question = Choice(
    instructions="영화 질문에 필요한 검색 경로를 선택하세요. 영화 제목 언급만으로 graph를 선택하지 마세요.",
    criteria={
        "hybrid": "배우 관계나 개봉연도 조건 없이 줄거리·사건·분위기로 영화를 찾거나 원문 내용을 묻는 질문",
        "graph": "줄거리 조건 없이 배우의 출연·공유 배우·개봉연도 등 그래프 관계나 속성만 조회하는 질문",
        "combined": "줄거리·사건·분위기 조건과 배우 관계·개봉연도 등 그래프 조건을 함께 만족해야 하는 질문",
    },
)


### 🖐️ 함께 따라하기: 원 질문을 두 검색 입력으로 나누는 체인을 만듭니다

**제공 코드:** `SearchQueries` 출력 형식과 `query_split_prompt` 전체를 제공합니다. 줄거리와 그래프 조건을 분리하는 지시를 읽어 보세요.

**작성할 부분:**

- `query_split_chain`은 `query_split_prompt`와 `llm.with_structured_output(SearchQueries)`를 `|`로 연결하세요.


In [ ]:
# 검색별 입력을 정해진 두 필드로 받습니다.
class SearchQueries(BaseModel):
    """원 질문에서 분리한 줄거리 검색어와 그래프 조건 질문입니다."""
    semantic_query: str = Field(description="전문·벡터 검색에 사용할 줄거리의 사건·상황")
    graph_request: str = Field(description="질문에 명시된 모든 그래프 관계·속성 조건을 담은 자연어 질문")


# 줄거리는 내용 검색에, 관계·속성 조건은 그래프 검색에 전달합니다.
query_split_prompt = ChatPromptTemplate.from_messages([
    ("system", "영화 질문을 두 검색 입력으로 나누세요. "
               "semantic_query에는 줄거리의 사건·상황만 담고 그래프 관계·속성 조건은 빼세요. "
               "graph_request에는 질문에 명시된 모든 그래프 관계·속성 조건과 조회 대상을 보존하세요. 예를 들어 출연·감독·제작 관계, 개봉연도, 인원 수, 제외 대상입니다. "
               "graph_request에는 줄거리 조건을 넣지 말고 Cypher가 아닌 자연어 질문을 작성하세요. "
               "숫자와 이상·초과·이전·이후 등 범위 표현을 바꾸지 마세요. "
               "원 질문에 없는 조건·영화명·인물명을 추가하지 마세요."),
    ("human", "{question}"),
])
# ====================================================================
query_split_chain = ...
# ====================================================================


### 🖐️ 함께 따라하기: 일반화된 Text-to-Cypher 체인을 만듭니다

**제공 코드:** `cypher_prompt`의 전체 문자열과 두 모드의 `RETURN` 형식을 제공합니다. `{schema}`는 체인이 DB 스키마로 자동으로 채웁니다. 복합 검색의 Cypher는 관계·속성 조건만 조회합니다. 줄거리는 전문·벡터 검색이 처리하므로 원문에 특정 단어가 반드시 있어야 한다는 조건을 만들지 않습니다.

**작성할 부분:**

- `graph`를 `Neo4jGraph`로 만드세요. `url=neo4j_uri`, 환경변수 `NEO4J_USER`·`NEO4J_PASSWORD`, `enhanced_schema=False`를 사용하세요.
- `graph_chain`을 `GraphCypherQAChain.from_llm`으로 만드세요. `llm`·`graph`·`cypher_prompt`를 연결하고, `include_types`에는 `Movie`, `Person`, `Document`, `ACTED_IN`, `DIRECTED`, `PRODUCED`, `WROTE`, `REVIEWED`, `FOLLOWS`, `ABOUT_MOVIE`를 넣으세요.
- `return_direct=True`, `return_intermediate_steps=True`, `top_k=100`, `allow_dangerous_requests=True`를 지정하세요. 조회 행과 생성 Cypher를 함께 받습니다. 마지막 옵션은 생성 쿼리 실행에 동의하는 설정이며 DB 쓰기 권한을 제한하지는 않습니다.


In [ ]:
# 이 실습에서 적재한 영화·인물·원문과 관계의 스키마를 사용합니다.
# ====================================================================
graph = ...
# ====================================================================
cypher_prompt = PromptTemplate.from_template("""
주어진 스키마와 질문으로 조회용 Cypher 하나만 반환하세요.
검색 모드: {search_mode}
공통 규칙과 현재 검색 모드의 규칙을 함께 적용하세요.

[공통 규칙]
- 데이터를 변경하지 말고, 스키마의 레이블·속성·관계 방향을 따르세요.
- 질문에 명시된 그래프 대상·관계·속성·숫자·범위 조건을 보존하고, 없는 조건을 추가하지 마세요.
- 개수 조건은 WITH에서 count(DISTINCT 대상) AS 별칭으로 집계한 뒤 WHERE에서 별칭과 기준값을 비교하세요. WHERE에 count()를 직접 쓰지 마세요.
- 이후 절에서 쓸 변수와 facts에 필요한 속성은 각 WITH에 변수 또는 별칭으로 전달하세요.

[공통 반환값]
- facts: 질문에 필요한 이름·제목·속성값·집계값을 담은 맵. 노드·관계 객체나 evidence 원문은 넣지 마세요.
- claim_ids: 답의 근거로 사용한 모든 관계의 실제 claim_id 목록. 중복을 제거하고, 없으면 []를 반환하세요.

[graph 모드]
- 그래프 관계·속성을 조회하고 다음 두 열만 반환하세요.
RETURN facts AS facts, claim_ids AS claim_ids LIMIT 100

[combined 모드]
- 그래프 관계·속성 조건에 맞고 Document 원문이 연결된 후보를 조회하세요.
- 줄거리·사건·분위기는 전문·벡터 검색이 처리합니다. 질문에 포함돼도 Cypher 조건으로 만들지 마세요.
- 줄거리를 Document.text·Chunk.text의 CONTAINS·정규식이나 추측한 그래프 속성으로 필터링하지 마세요.
- Document.id를 source_id로 사용해 원문별 한 행, 다음 세 열만 반환하세요.
RETURN d.id AS source_id, facts AS facts, claim_ids AS claim_ids ORDER BY source_id LIMIT 100

[스키마]
{schema}

[질문]
{question}
""")
# ====================================================================
graph_chain = ...
# ====================================================================


### 🖐️ 함께 따라하기: 원 질문의 경로를 판단하고 검색 입력을 준비합니다

**제공 코드:** 예시 질문 `new_question`, 복합 경로의 분기와 결과 출력은 제공합니다.

**작성할 부분:**

- `route_decision`에 `route_classifier.invoke`의 결과를 담으세요. `state`는 `new_question`, `questions`는 `{"route": route_question}`입니다.
- `new_route`에 `route_decision.choices["route"].choice`를 담으세요.
- `combined` 분기에서 `query_split_chain`에 `{"question": new_question}`을 전달해 `new_queries`를 만드세요.


In [ ]:
# 질문은 여기서 한 번 정의하고 최종 답변에도 그대로 사용합니다.
new_question = "Sleepless in Seattle과 출연 배우를 최소 2명 공유하고 1993년 이후(1993년 제외) 개봉한 다른 영화 중 서로 경쟁하는 서점 주인들이 익명 이메일로 사랑하게 되는 작품을 찾아 공통 배우와 근거를 보여 주세요."
# Jev가 질문 유형에 맞는 경로 하나를 선택합니다.
# ====================================================================
route_decision = ...
# ====================================================================
# ====================================================================
new_route = ...
# ====================================================================
print("검색 경로:", new_route)

if new_route == "combined":
    # 줄거리와 그래프 조건을 모두 묻는 경우에만 질문을 분리합니다.
    # ====================================================================
    new_queries = ...
    # ====================================================================
    print("줄거리 검색어:", new_queries.semantic_query)
    print("그래프 조건 질문:", new_queries.graph_request)


### 🖐️ 함께 따라하기: 선택한 경로로 검색하고 근거를 모읍니다

**제공 코드:** 경로 분기·검색어 선택·RRF 결과 변환·그래프 조건 적용·후보 보완·근거 수집·출력은 제공합니다.

**작성할 부분:**

- `graph` 분기의 `new_graph_result`는 `graph_chain.invoke`로 만드세요. `query`는 `new_question`, `search_mode`는 `"graph"`입니다.
- `new_docs`는 `ensemble_retriever.invoke(new_semantic)`로 전문·벡터 검색 결과를 받으세요.
- `combined` 분기의 `new_graph_result`도 `graph_chain.invoke`로 만드세요. `query`는 `new_queries.graph_request`, `search_mode`는 `"combined"`입니다.


In [ ]:
if new_route == "graph":
    # 그래프 질문은 전문·벡터 검색이나 질문 임베딩을 호출하지 않습니다.
    # ====================================================================
    new_graph_result = ...
    # ====================================================================
    print(new_graph_result["intermediate_steps"][0]["query"])
    new_graph_rows = new_graph_result["result"]
    display(pd.DataFrame(new_graph_rows))
    # 조회 사실과 실제 관계 근거를 함께 전달합니다.
    new_evidence = collect_evidence(new_graph_rows)
else:
    # 일반 질문은 원문 그대로, 복합 질문은 분리한 줄거리로 하이브리드 검색합니다.
    new_semantic = new_queries.semantic_query if new_route == "combined" else new_question
    # ====================================================================
    new_docs = ...
    # ====================================================================
    # RRF가 정한 순서를 유지하며 원문 ID만 사용합니다.
    new_fused = [{"source_id": doc.metadata["source_id"]} for doc in new_docs]

    if new_route == "combined":
        # 복합 경로는 그래프 조건에 맞는 후보만 남기고 조회 근거를 함께 모읍니다.
        # ====================================================================
        new_graph_result = ...
        # ====================================================================
        print(new_graph_result["intermediate_steps"][0]["query"])
        new_graph_rows = new_graph_result["result"]
        new_allowed = {row["source_id"] for row in new_graph_rows}
        new_candidates = [row for row in new_fused if row["source_id"] in new_allowed]
        new_fused_ids = {row["source_id"] for row in new_fused}
        new_candidates += [{"source_id": source_id}
                           for source_id in sorted(new_allowed - new_fused_ids)]
        new_top_ids = [row["source_id"] for row in new_candidates[:3]]
        new_selected_graph = [row for row in new_graph_rows if row["source_id"] in new_top_ids]
        new_evidence = collect_evidence(new_selected_graph, new_top_ids)
    else:
        # 일반 경로는 검색 순위를 유지하고 줄거리 원문만 전달합니다.
        new_candidates = new_fused
        new_top_ids = [row["source_id"] for row in new_candidates[:3]]
        new_evidence = collect_evidence([], new_top_ids)

    print("통합 검색 문서:", len(new_fused), "최종 후보:", len(new_candidates))
    show_documents(new_candidates)


### 🖐️ 함께 따라하기: 검색 근거로 답하는 체인을 만듭니다

아래 코드를 그대로 실행하세요. `SearchAnswer` 출력 형식과 `search_answer_prompt`의 인용 지시문 전체를 제공합니다.


In [ ]:
# 세 경로 모두 질문에 필요한 내용과 실제 근거 ID만 반환합니다.
class SearchAnswer(BaseModel):
    """검색 근거에 기반한 답변과 인용 ID 목록입니다."""
    # 본문 인용을 빠뜨리지 않도록 대괄호 ID 형식도 출력 스키마에 지정합니다.
    answer: str = Field(pattern=r"\[[A-Za-z0-9:_-]+\]|근거가 부족합니다", description="각 주장 뒤에 [실제 evidence_id]를 포함한 간결한 한국어 답변")
    evidence_ids: list[str] = Field(description="답변 본문에 인용한 모든 근거 ID를 중복 없이 담은 목록")


search_answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "제공된 근거만 사용해 한국어로 간결하게 답하세요. "
               "각 주장 끝에 실제 근거 ID를 [ID]로 표시하고, 공유 관계는 양쪽 관계 근거를 인용하세요. "
               "ID는 접두어를 포함한 evidence_id 전체를 그대로 복사하고 대괄호 안에 공백을 넣지 마세요. "
               "evidence_ids에는 본문에 쓴 ID만 중복 없이 담으세요. "
               "근거가 없으면 '근거가 부족합니다'라고 답하고 인용 목록을 비우세요."),
    ("human", "질문: {question}\n근거: {context}"),
])
search_answer_chain = search_answer_prompt | llm.with_structured_output(SearchAnswer, strict=True)


### 검색 근거로 최종 답변을 요청합니다

아래 코드를 그대로 실행하세요. 답변 출력, 본문·목록의 인용 ID 대조와 인용 근거 표시는 제공합니다.

ID 검사는 인용의 존재 여부를 확인합니다. 근거가 주장을 뒷받침하는지도 직접 읽어 보세요.


In [ ]:
# 원 질문과 선택한 경로에서 얻은 근거로 한 번 답변합니다.
new_answer = search_answer_chain.invoke({"question": new_question, "context": json.dumps(new_evidence, ensure_ascii=False)})
print(new_answer.answer)
print(new_answer.evidence_ids)
# 인용 ID의 존재 여부와 본문 표시를 확인합니다.
new_evidence_by_id = {item["evidence_id"]: item for item in new_evidence}
new_inline_ids = set(re.findall(r"\[([^\[\]]+)\]", new_answer.answer))
new_unknown_ids = (new_inline_ids | set(new_answer.evidence_ids)) - set(new_evidence_by_id)
# ^는 본문과 인용 목록 중 한쪽에만 있는 근거 ID를 고릅니다.
new_citation_gap = new_inline_ids ^ set(new_answer.evidence_ids)
print("본문과 목록에 인용 있음:", bool(new_inline_ids) and bool(new_answer.evidence_ids))
print("제공하지 않은 인용 ID:", new_unknown_ids)
print("본문과 목록의 차이:", new_citation_gap)
for evidence_id in new_answer.evidence_ids:
    if evidence_id in new_evidence_by_id:
        print(evidence_id, new_evidence_by_id[evidence_id]["text"])


### 실행 결과를 확인합니다

3절 처음의 질문들로 분류 결과·생성 Cypher·최종 답변의 근거를 비교하세요.
그래프 단독 경로는 질문 임베딩·전문 검색·질문 분리를 호출하지 않습니다.
일반 경로는 Text-to-Cypher를 호출하지 않습니다. 복합 경로는 분리한 조건이 Cypher에 모두 반영됐는지 확인하세요.
